# 05 train model
1. Research candidates and evaluate approaches
2. Establish a baseline model
3. Implement the most promising candidates
4. Evaluate implementations
5. Select production candidate

# Research:

## Candidate #1
### TF-IDF + Logistic Regression
##### TF-IDF:
Feature extraction technique used in NLP to transform raw text into a numeric value that ML models can process/ understand.
The numerical value is a weight to each word based on two factors:
1. How often a word appears in a document/ record (TF, Term frequency)
2. How rare the word is across all documents (IDF, Inverse document frequency)

This helps the model to focus on words that actually inform it on differences between records by stripping away common unhelpful words and highlighting distinctive words in each record so the model can learn differences.

Reference: https://scikit-learn.org/1.4/tutorial/text_analytics/working_with_text_data.html

##### Logistic regression:
Linear classification algorithm used for both binary and multi-class classification problems. Since our problem is multi-class (i.e. we have more than two possible outcomes/ labels) we can use scikit-learn multinominal logistic regression to estimate the probability of each possible class using a softmax function.

Using the TF-IDF transformed data, the model can then learn a weight for each feature (a word) and an associated class. We then select the class with the highest probability as the prediction.

During inference the model calculates the probability for each possible resolver group and the class with the highest probability is selected as the final predicted value.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

##### Overview of combination
TF-IDF + Logistic regression will be our baseline as it is computationally efficient, simple to explain and interpret and produces a probabalistic output to our multi-class classification problem.

Pros:
- Simple... basically text input, TF-IDF, Logistic regression = predicted resolver group. Low number of steps makes it easy to explain and implement
- Quick to train when compared with more complex neural network models so we can quickly experiment and iterate
- No black box processing, we can actually view the feature weights and see what words appear to be influencing classification decisions
- Probabilistic output allows us to route incidents for manual inspection if the confidence is low

Cons:
- TF-IDF just captures individual words, losing the meaning and the context of the words in the record
- Becuase each word is treated as a feature, the feature matrices can grow fairly large on a large number of records with lots of distinct words

Summary:
TF-IDF and logistic regression provide a good baseline which we can interpret and inspect as humans to understand it's behaviour. Where we may run into issues is the loss of context and meaning since TF-IDF treats individual words as features instead of phrases etc, we may find the model struggles with less obvious text descriptions or a large amount of variance in the words used for incidents of the same class.

To overcome these cons, other candidates will consider more advanced transformer based models.

____________________________________________________________________________________________________________________________________________

## Candidate #2
### DistilBERT - Transformer based language model
##### Transformer element
Transformers are a type of neural network design for processing natural language. The main difference between transformers and traditional NLP techniques like TF-IDF is that instead of treating individual words independently, transformers actually analyse the relationship between words in a sentence/ body of text using a mechanism called self-attention. This enables the model to understand not just words but the actual meaning of words in context.

References: 
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://www.ibm.com/think/topics/self-attention

##### DistilBERT
DistilBERT is a smaller and faster version of the BERT transformer model. It was created using a technique called knowledge distillation, which is where a smaller model is trained to replicate the behaviour of a larger pre-trained model. The result of that is a model that maintains a lot of the predictive power of the parent model but uses significantly less parameters and provides faster inference. This makes DistilBERT a good option when considering trade offs between performance/ computational cost.

References: 
- https://huggingface.co/docs/transformers/model_doc/bert
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://towardsdatascience.com/everything-you-need-to-know-about-albert-roberta-and-distilbert-11a74334b2da/

##### Overview of combination
DistilBERT can be tuned for a sequence classification task where the model directly predicts the resolver group based on the text input. Essentially text input, tokenizer, DistilBERT, classification head = predicted resolver group. The model outputs a probabilistic output similar to candidate #1, allowing us to select the highest probability as the class prediction.

Pros:
- Captures relationship between words to better understand context/ meaning
- Able to better understand variations in wording for the same incident type
- Often performs better on text classification tasks vs basic classifiers

Cons:
- Significantly more computationally expensive than traditional NLP models
- Requires additional dependencies like transformers and other libraries
- Harder to interpret compared with a linear model due to the complexity of the neural network

Summary:
DistilBERT will provide us with a more advanced approach to text classification by capturing relationships between words instead of just the individual words alone. This should in theory mean improved classification performance when the words used in descriptions vary. However, it will add complexity, longer training times and additional requirements for libraries/ infrastructure which can all be negatives.

____________________________________________________________________________________________________________________________________________

## Candidate #3
### SetFit - sentence transformer + classifier
##### Sentence transformer
Sentence transformers convert text into vector representations aka embeddings. These embeddings represent the meaning of a sentence rather than individual words, similar to candidate #2. This enables sentences with similar meanings to produce similar vector repsresentations so a model can make that link and use it to help classify a record.

References: 
- https://huggingface.co/docs/setfit/index
- https://sbert.net/index.html

##### SetFit
SetFit is a framework that combines sentence transformers with a classifier. Instead of full tuning a transformer model, setfit generates embeddings using a pre-trained transformer model and then trains a smaller classifier on top of those embeddings. This way it can reduce training time whilst still getting the benefit of the context and understanding provided by transformer models.

References: 
- https://huggingface.co/docs/setfit/index
- https://medium.com/@youssefchamrah/setfit-unpacked-when-sentence-transformers-go-to-gym-for-classification-muscle-56c16d9e69de

##### Overview of combination
SetFit looks simple when looking at it process-wise due to it's pre-trained nature. text input, sentence embeddings, classifier = predicted resolver group. It is also significantly faster than tuning a transformer model, again due to it's pre-trained nature.

Pros:
- Captures context and meaning using transformer embeddings
- Faster training compared with a full transformer tuning
- Lower computational cost than models like candidate #2

Cons:
- Introduces another modelling framework and dependency to the solution
- Less common to find implementations/ examples and research than the other candidates
- If there is lots of labelled training data, the benefit of the pre-trained nature is lowered significantly

Summary:
SetFit fits somewhere betweeen candidate #1 a traditional ML approach and candidate #2 a transformer based model. It can provide us with context and meaning similar to the transformer models whilst keeping training time lower due to it being pre-trained. However, if there is lots of training data which is labelled with the target we are trying to predict, it's benefit can be lessened significantly.

____________________________________________________________________________________________________________________________________________